# NLP / CV 小考整合 Notebook（Colab）
專案結構：`main.ipynb`、`Codes/Quiz1.py ~ Quiz4.py`、`Data/`、`Outputs/`
- 圖片輸出：`Outputs/Q{n}_*.png`
- 模型輸出：`Outputs/q{n}_*.pt`

**執行前**：把 `ROOT` 改成你在 Google Drive 的專案資料夾，並把 `Quiz1.py ~ Quiz4.py` 放進 `ROOT/Codes/`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 修改成你的專案資料夾
ROOT = '/content/drive/MyDrive/Quizzes'

In [ ]:
!pip -q install medmnist datasets bert_score sacrebleu google-genai transformers

In [ ]:
import sys, os
import numpy as np, pandas as pd
import torch

sys.path.append(f'{ROOT}/Codes')
import Quiz1 as q1
import Quiz2 as q2
import Quiz3 as q3
import Quiz4 as q4

paths = q1.get_paths(ROOT)
q1.set_seed(42)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## Quiz 1：資料集準備

In [ ]:
imdb_train, imdb_test = q1.prepare_imdb(paths)

In [ ]:
blood_info = q1.prepare_bloodmnist(paths, size=224)

In [ ]:
# 第一次執行會把 8000 張圖寫進 Drive，需數分鐘
flickr = q1.prepare_flickr8k(paths)

## Quiz 2：Sentiment Analysis（RNN vs LSTM）

In [ ]:
ctx2 = q2.prepare_data(paths, imdb_train, imdb_test, vocab_size=20000)
MAX_LEN = 256

### 2-1 RNN

In [ ]:
rnn_res = q2.run_experiment(ctx2, 'rnn', MAX_LEN, epochs=8)
q2.save_model(ctx2, rnn_res[0], 'rnn', MAX_LEN)

### 2-2 LSTM

In [ ]:
lstm_res = q2.run_experiment(ctx2, 'lstm', MAX_LEN, epochs=8)
q2.save_model(ctx2, lstm_res[0], 'lstm', MAX_LEN)

### 2-3 測試集推論與分類結果

In [ ]:
rnn_model, _, rnn_p, test_y, _ = rnn_res
lstm_model, _, lstm_p, _, _ = lstm_res
pred_df = q2.save_test_predictions(ctx2, test_y, rnn_p, lstm_p)
pred_df.sample(10, random_state=1)

In [ ]:
demo = ["This movie was absolutely wonderful, I loved every minute of it.",
        "A boring, predictable mess. I want my two hours back.",
        "The first half was slow but the ending was brilliant and moving."]
print(pd.DataFrame({'text': demo,
                    'RNN_pos_prob': q2.predict_texts(ctx2, rnn_model, demo, MAX_LEN),
                    'LSTM_pos_prob': q2.predict_texts(ctx2, lstm_model, demo, MAX_LEN)}).round(3).to_string())

### 2-4 RNN vs LSTM 評估比較

In [ ]:
metrics2 = q2.compare(ctx2, rnn_res, lstm_res)
display(metrics2.round(4))

### 2-5（加分）不同序列長度對照實驗（共 8 次訓練，時間有限可略過）

In [ ]:
RUN_LENGTH_EXP = True
if RUN_LENGTH_EXP:
    len_df = q2.length_experiment(ctx2, lengths=(64, 128, 256, 400), epochs=5)
    display(len_df.round(4))

## Quiz 3：Vision Transformer vs ResNet（BloodMNIST 224）

In [ ]:
ctx3 = q3.build_context(paths, blood_info['class_names'], bs=64)

### 3-1 微調 ViT-B/16

In [ ]:
vit_res = q3.train(ctx3, 'vit', epochs=5, lr=3e-5)

### 3-2 微調 ResNet-50

In [ ]:
resnet_res = q3.train(ctx3, 'resnet', epochs=5, lr=1e-4)

### 3-3 測試集推論與 Macro-AUC 比較

In [ ]:
res3, per_cls3, probs3, y3 = q3.evaluate_and_plot(ctx3, vit_res, resnet_res)
display(res3.round(4))
display(per_cls3.round(4))

In [ ]:
q3.show_predictions(ctx3, probs3, n=10)

## Quiz 4：Image Captioning
三種方法（batch size = 1 計時）：A 自訓練 CNN+LSTM、B BLIP、C Gemini。

In [ ]:
# ---- 設定 ----
N_EVAL = 100     # 三種方法共同評估的測試圖張數
N_JUDGE = 50     # 送 Gemini 評判的張數（受免費額度限制）
GEMINI_MODEL = 'gemini-2.5-flash'   # 若不可用，請改成目前可用的 Gemini 模型
GEMINI_SLEEP = 4                    # 每次呼叫後等待秒數（避免超過速率限制）

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')   # 在 Colab 左側「密鑰」新增
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

ctx4 = q4.load_context(paths)
q4.setup_gemini(ctx4, GEMINI_API_KEY, GEMINI_MODEL, GEMINI_SLEEP)

### 4-1 抽取 ResNet-50 特徵（快取）

In [ ]:
q4.extract_features(ctx4)

### 4-2 訓練 CNN+LSTM 解碼器（方法 A）

In [ ]:
q4.build_vocab(ctx4, min_freq=5)
cap_hist = q4.train_decoder(ctx4, epochs=25, patience=3)

### 4-3 三種方法產生描述

In [ ]:
q4.load_blip(ctx4)
q4.build_eval_set(ctx4, n_eval=N_EVAL)
q4.generate_all(ctx4)

### 4-4 BLEU / BERTScore

In [ ]:
metric_df = q4.text_metrics(ctx4)
metric_df

### 4-5 Gemini 判斷描述是否符合影像

In [ ]:
if ctx4.get('has_gemini'):
    judge_summary = q4.judge_all(ctx4, n_judge=N_JUDGE)
    metric_df = metric_df.join(judge_summary)
    display(ctx4['judge_df'].head(10))

### 4-6 實驗結果整理

In [ ]:
final_df = q4.summarize_and_plot(ctx4, metric_df, n_judge=N_JUDGE)
display(final_df)

## 結論（請依實際結果填寫）
- **Quiz 2**：比較 RNN / LSTM 的 Accuracy、Macro-F1、AUC 與訓練時間；參考 `Q2_length_experiment.png` 說明序列長度的影響。
- **Quiz 3**：比較 ViT 與 ResNet 的 Macro-AUC、每類別 AUC 與訓練成本。
- **Quiz 4**：比較三種方法在 BLEU / BERTScore / Gemini 判斷 / 延遲上的取捨（Gemini 評判自己的描述可能有偏袒，需註明）。

In [ ]:
for p in sorted(paths['outputs'].iterdir()):
    print(p.name)